LR หรือ Logistic Regression เป็นโมเดลสำหรับทำนายผลออกมาเป็นในเชิง True false เช่น คุณเป็นมะเร็งหรือไม่ คุณมีเเนวโน้มที่จะเป็นโควิดหรือไม่

In [58]:
import pandas as pd
import joblib 
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


In [59]:
#Load Dataset

dataset = pd.read_csv("../../data/processed/Titanic-Dataset.csv")

In [60]:
target = "Survived"

x = dataset.drop(columns=[target, "PassengerId", "Name", "Ticket"])

# Cabin
x["Cabin"] = x["Cabin"].str[0]
x["Cabin"] = x["Cabin"].fillna("Unknown")

# Sex
x["Sex"] = x["Sex"].map({"male": 1, "female": 0})

# Embarked
x["Embarked"] = x["Embarked"].fillna("Unknown")
x = pd.get_dummies(x, columns=["Embarked", "Cabin"])

# Age
x["Age"] = x["Age"].fillna(x["Age"].median())

# fill numeric
x = x.fillna(x.mean(numeric_only=True))

# convert type
x = x.astype(float)

y = dataset[target]

print(x.head())

   Pclass  Sex   Age  SibSp  Parch     Fare  Embarked_C  Embarked_Q  \
0     3.0  1.0  22.0    1.0    0.0   7.2500         0.0         0.0   
1     1.0  0.0  38.0    1.0    0.0  71.2833         1.0         0.0   
2     3.0  0.0  26.0    0.0    0.0   7.9250         0.0         0.0   
3     1.0  0.0  35.0    1.0    0.0  53.1000         0.0         0.0   
4     3.0  1.0  35.0    0.0    0.0   8.0500         0.0         0.0   

   Embarked_S  Embarked_Unknown  Cabin_A  Cabin_B  Cabin_C  Cabin_D  Cabin_E  \
0         1.0               0.0      0.0      0.0      0.0      0.0      0.0   
1         0.0               0.0      0.0      0.0      1.0      0.0      0.0   
2         1.0               0.0      0.0      0.0      0.0      0.0      0.0   
3         1.0               0.0      0.0      0.0      1.0      0.0      0.0   
4         1.0               0.0      0.0      0.0      0.0      0.0      0.0   

   Cabin_F  Cabin_G  Cabin_T  Cabin_Unknown  
0      0.0      0.0      0.0            1.0  


In [61]:
scaler = StandardScaler()
x = scaler.fit_transform(x)


x_train , x_test , y_train , y_test = train_test_split(x , y, test_size=0.2)


x_train = scaler.fit_transform(x_train)
x_test  = scaler.transform(x_test)

X_train_tensor = torch.tensor(x_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)

X_test_tensor = torch.tensor(x_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

In [72]:
model = nn.Sequential(
    nn.Linear(X_train_tensor.shape[1], 16),
    nn.GELU(),
    nn.Linear(16, 8),
    nn.GELU(),
    nn.Linear(8, 1),
    nn.Sigmoid()
)

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(800):

    # forward
    y_pred = model(X_train_tensor)

    # loss
    loss = criterion(y_pred, y_train_tensor)

    # backward
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item()}")

Epoch 0, Loss: 0.6966657042503357
Epoch 50, Loss: 0.6448061466217041
Epoch 100, Loss: 0.5383913516998291
Epoch 150, Loss: 0.4635073244571686
Epoch 200, Loss: 0.4360518157482147
Epoch 250, Loss: 0.42214086651802063
Epoch 300, Loss: 0.409236341714859
Epoch 350, Loss: 0.3966331481933594
Epoch 400, Loss: 0.3849862217903137
Epoch 450, Loss: 0.37412190437316895
Epoch 500, Loss: 0.3648219108581543
Epoch 550, Loss: 0.3566267788410187
Epoch 600, Loss: 0.3490936756134033
Epoch 650, Loss: 0.3419475555419922
Epoch 700, Loss: 0.3351464569568634
Epoch 750, Loss: 0.3283890187740326


In [ ]:
model.eval()

with torch.no_grad():
    y_pred = model(X_test_tensor)
    y_pred_class = (y_pred > 0.5).float()

    accuracy = (y_pred_class == y_test_tensor).sum() / len(y_test_tensor)

    print("Accuracy:", accuracy.item())

Accuracy: 0.8156424760818481


In [ ]:
joblib.dump(model , "titanic_NN.pkl")

['titanic_NN.pkl']